# ALS Tri-Stream Classifier — Training


## 1 · Imports

In [ ]:
import json

import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from torch.utils.data import DataLoader, Subset

from classifier import ALSTriStreamClassifier
from dataset import MultiModalALSDataset
from paths import CHECKPOINT_PATH, DATA_DIR, METRICS_DIR, ensure_output_dirs
from split_utils import split_indices_by_subject

## 2 · Hyperparameters

Edit these before running.

In [ ]:
SEED                 = 42
BATCH_SIZE           = 8        # RTX 5090 32 GB — comfortable at 128³
NUM_EPOCHS           = 100       # more epochs since backbone is frozen longer
LR_HEAD              = 3e-4
LR_BACKBONE          = 3e-5
WEIGHT_DECAY         = 1e-4
UNFREEZE_AFTER_EPOCH = 70 

DEVICE = torch.device(
    "mps"  if torch.backends.mps.is_available()  else
    "cuda" if torch.cuda.is_available()          else
    "cpu"
)

torch.manual_seed(SEED)
print(f"Device : {DEVICE}")
print(f"Data   : {DATA_DIR}")

## 3 · Dataset & splits

In [ ]:
ensure_output_dirs()

if not DATA_DIR.exists():
    raise FileNotFoundError(f"Data directory not found: {DATA_DIR}")

# Full dataset — no augmentation, used for val/test and split indices
# 128³ input: justified by 32 GB VRAM, improves receptive field coverage
fullDataset  = MultiModalALSDataset(rootDirectory=str(DATA_DIR), transform=False, targetShape=(128, 128, 128))
train_idx, val_idx, _ = split_indices_by_subject(
    fullDataset.samples, train_ratio=0.8, val_ratio=0.1, seed=SEED
)

if not train_idx:
    raise RuntimeError("Training set is empty — check your data directory.")

# Separate augmented instance for training split only
trainDataset = MultiModalALSDataset(rootDirectory=str(DATA_DIR), transform=True, targetShape=(128, 128, 128))
trainSet     = Subset(trainDataset, train_idx)
valSet       = Subset(fullDataset,  val_idx)

# num_workers=4 + pin_memory: overlaps data loading with GPU compute
trainLoader = DataLoader(trainSet, batch_size=BATCH_SIZE, shuffle=True,  num_workers=4, pin_memory=True, persistent_workers=True)
valLoader   = DataLoader(valSet,   batch_size=BATCH_SIZE, shuffle=False, num_workers=4, pin_memory=True, persistent_workers=True)

print(f"Train samples : {len(trainSet)}")
print(f"Val   samples : {len(valSet)}")

## 4 · Class balance & loss

In [ ]:
train_labels = [fullDataset.samples[i]["label"] for i in train_idx]
n_pos = sum(train_labels)
n_neg = len(train_labels) - n_pos
print(f"Controls : {int(n_neg)}")
print(f"ALS      : {int(n_pos)}")
print(f"pos_weight (neg/pos) : {n_neg/n_pos:.3f}" if n_pos > 0 else "WARNING: no positive samples")

# Weighted BCE to handle class imbalance
pos_weight = torch.tensor([n_neg / n_pos] if n_pos > 0 else [1.0], device=DEVICE)
criterion  = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

## 5 · Model, optimiser & scheduler

In [ ]:
model = ALSTriStreamClassifier(freeze_backbone=True).to(DEVICE)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f"Total params     : {total:,}")
print(f"Trainable params : {trainable:,}  ({100*trainable/total:.1f}%)")

optimizer = AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=LR_HEAD,
    weight_decay=WEIGHT_DECAY,
)
scheduler = CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS)

## 6 · Training loop

In [ ]:
best_val_loss     = float("inf")
history           = {"train_loss": [], "val_loss": [], "val_acc": []}
backbone_unfrozen = False

for epoch in range(1, NUM_EPOCHS + 1):

    # ── Phase 2: unfreeze backbone ────────────────────────────────────────
    if (
        UNFREEZE_AFTER_EPOCH is not None
        and epoch == UNFREEZE_AFTER_EPOCH + 1
        and not backbone_unfrozen
    ):
        print(f"\n[Epoch {epoch}] Unfreezing backbone for fine-tuning...")
        for param in model.parameters():
            param.requires_grad = True

        # Enable gradient checkpointing on all three ResNet50 backbones.
        # This trades ~30% extra compute for a ~40% reduction in activation
        # memory — essential when unfreezing 3x ResNet50 at batch_size=8.
        for encoder in (model.model.t1Encoder, model.model.t2Encoder, model.model.flairEncoder):
            if hasattr(encoder._backbone, 'set_grad_checkpointing'):
                encoder._backbone.set_grad_checkpointing(True)
            elif hasattr(encoder._backbone, 'gradient_checkpointing_enable'):
                encoder._backbone.gradient_checkpointing_enable()

        # Reduce batch size for Phase 2 — unfrozen ResNet50 x3 needs more
        # activation memory for backprop. Recreate loaders with batch_size=4.
        BATCH_SIZE_P2 = 4
        trainLoader = DataLoader(
            trainSet, batch_size=BATCH_SIZE_P2, shuffle=True,
            num_workers=4, pin_memory=True, persistent_workers=True
        )
        valLoader = DataLoader(
            valSet, batch_size=BATCH_SIZE_P2, shuffle=False,
            num_workers=4, pin_memory=True, persistent_workers=True
        )
        print(f"  Batch size reduced to {BATCH_SIZE_P2} for Phase 2.")

        optimizer = AdamW(model.parameters(), lr=LR_BACKBONE, weight_decay=WEIGHT_DECAY)
        scheduler = CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS - epoch)
        backbone_unfrozen = True

    # ── Train ─────────────────────────────────────────────────────────────
    model.train()
    train_loss = 0.0

    for inputs, labels in trainLoader:
        t1, t2, flair = [v.to(DEVICE) for v in inputs]
        labels        = labels.to(DEVICE)

        optimizer.zero_grad()
        logits = model(t1, t2, flair).squeeze(1)      # (B,)
        loss   = criterion(logits, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        train_loss += loss.item()

    train_loss /= max(len(trainLoader), 1)

    # ── Validate ──────────────────────────────────────────────────────────
    model.eval()
    val_loss = 0.0
    correct  = 0
    total    = 0

    with torch.no_grad():
        for inputs, labels in valLoader:
            t1, t2, flair = [v.to(DEVICE) for v in inputs]
            labels        = labels.to(DEVICE)

            logits    = model(t1, t2, flair).squeeze(1)
            val_loss += criterion(logits, labels).item()
            preds     = (torch.sigmoid(logits) >= 0.5).long()
            correct  += (preds == labels.long()).sum().item()
            total    += labels.size(0)

    val_loss /= max(len(valLoader), 1)
    val_acc   = correct / total if total > 0 else 0.0
    scheduler.step()

    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)
    history["val_acc"].append(val_acc)

    print(
        f"Epoch {epoch:>3}/{NUM_EPOCHS} | "
        f"train_loss={train_loss:.4f} | "
        f"val_loss={val_loss:.4f} | "
        f"val_acc={val_acc:.3f}"
    )

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), CHECKPOINT_PATH)
        print(f"  ✓ Checkpoint saved (val_loss={best_val_loss:.4f})")

print(f"\nTraining complete. Best val_loss: {best_val_loss:.4f}")

## 7 · Save training history

In [ ]:
history_path = METRICS_DIR / "training_history.json"
with history_path.open("w") as f:
    json.dump(history, f, indent=2)

print(f"Checkpoint : {CHECKPOINT_PATH}")
print(f"History    : {history_path}")

## 8 · Plot training curves

In [ ]:
epochs = range(1, len(history["train_loss"]) + 1)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(epochs, history["train_loss"], label="Train loss")
ax1.plot(epochs, history["val_loss"],   label="Val loss")
if UNFREEZE_AFTER_EPOCH is not None and UNFREEZE_AFTER_EPOCH < len(epochs):
    ax1.axvline(x=UNFREEZE_AFTER_EPOCH, color="grey", linestyle="--", label="Backbone unfrozen")
ax1.set_xlabel("Epoch")
ax1.set_ylabel("BCE Loss")
ax1.set_title("Loss")
ax1.legend()
ax1.grid(True, alpha=0.3)

ax2.plot(epochs, history["val_acc"], color="tab:green", label="Val accuracy")
if UNFREEZE_AFTER_EPOCH is not None and UNFREEZE_AFTER_EPOCH < len(epochs):
    ax2.axvline(x=UNFREEZE_AFTER_EPOCH, color="grey", linestyle="--", label="Backbone unfrozen")
ax2.set_xlabel("Epoch")
ax2.set_ylabel("Accuracy")
ax2.set_title("Validation Accuracy")
ax2.set_ylim(0, 1)
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plot_path = METRICS_DIR / "training_curves.png"
plt.savefig(plot_path, dpi=150)
plt.show()
print(f"Saved to {plot_path}")